# SDAR Study Tour

Learning map for **Self-Distilled Agentic Reinforcement Learning** (Lu et al. 2026, arxiv:2605.15155).

This notebook mirrors `tour.md` with executable cells for the measurement-style improvements (102, 103).


## 1. Reader's contract

Three skill-level entry points: **Beginner** (6-8h, all of Section 2), **Intermediate** (3-4h, focus Section 3), **Advanced** (1-2h, jump to Section 3 and 4). Full chain prerequisite: Chain Ch 25, 27, 28, 30, 31. Companion artifacts: 14 paper concept notes, 4 improvement concept notes, 18 runnable Python demos, 2 LaTeX proofs.


## 2. Foundations walk

- **Ch 25 — Causal LM as MDP.** Per-token autoregressive decoding as an MDP. _30 min._
- **Ch 27 — Tiny GPT pre-training.** Decoder-only Transformer architecture. _60 min, mostly review._
- **Ch 28 — SFT, RLHF, DPO.** KL-to-reference is the same primitive SDAR uses. _90 min, central._
- **Ch 30 — Maximum-entropy RL.** Lagrangian dual interpretation; SDAR is a Lagrangian-style auxiliary. _60 min._
- **Ch 31 — PG / GRPO / RLHF-DPO bridge.** SDAR's RL backbone is GRPO verbatim from this chapter. _90-120 min, core._


## 3. Paper concepts walk

Topological order for the 14 concept notes:

1. Multi-turn agentic MDP
2. Trajectory-level RL reward
3. GRPO baseline
4. OPSD (on-policy self-distillation)
5. Teacher branch + privileged context
6. Per-token reverse KL
7. Single-sample gap estimator $\Delta_t$
8. Multi-turn instability (Obs 1)
9. Asymmetric trust (Obs 2)
10. Sigmoid token-level gate (stop-gradient)
11. Entropy gating $g_t = \sigma(\beta h_t)$
12. Gap gating $g_t = \sigma(\beta \Delta_t)$ — centerpiece
13. Soft-OR gating
14. SDAR combined objective $\mathcal{L} = \mathcal{L}_{GRPO} + \lambda_{SDAR} \mathcal{L}_{SDAR}$


## 4. Improvements walk

- **101 — Bias bound for gap gating** (PROOF, see `proofs/gating-bias-bound.tex`). Sub-Gaussian gap → bias $\leq \tfrac{1}{2}E|\Delta_t| + O(\beta\sigma_\Delta^2)$.
- **102 — Adaptive gating threshold** (MEASUREMENT). Code cell below.
- **103 — Lambda sensitivity sweep** (MEASUREMENT). Code cell below.
- **104 — SDAR as constrained RL** (PROOF, see `proofs/sdar-as-constrained-rl.tex`). $\lambda_{SDAR}$ as Lagrange multiplier; primal-dual convergence transfers from CMDP literature.


### 4a. Improvement 102 — adaptive gating measurement

Imports the `measure()` function from the sandbox script that Agent C is writing under `improvements/adaptive-gate.py`. The signature returned by `measure()` is a dict of summary statistics (firing fraction, gate variance, etc.) per epoch.


In [ ]:
import sys, os
from pathlib import Path

# Locate the sandbox improvements directory (sibling of learning-map/)
study_dir = Path(os.environ.get('SDAR_STUDY_DIR',
    '/home/pleyv/ai-research-studies/self-distilled-agentic-reinforcement-learning'))
improvements_dir = study_dir / 'improvements'
sys.path.insert(0, str(improvements_dir))

try:
    from importlib import import_module
    adaptive = import_module('adaptive_gate' if (improvements_dir / 'adaptive_gate.py').exists()
                             else 'adaptive-gate')
    result = adaptive.measure()
    print('adaptive-gate measure() ->', result)
except Exception as e:
    print(f'(Agent C has not yet written adaptive-gate.py: {e})')
    print('Falling back to inline demo of improvement 102:')
    exec(open(study_dir / 'learning-map/improvements/code/102-adaptive-gating-threshold.py').read())


### 4b. Improvement 103 — lambda sensitivity measurement

Imports `measure()` from `improvements/lambda-sweep.py`.


In [ ]:
try:
    lam = import_module('lambda_sweep' if (improvements_dir / 'lambda_sweep.py').exists()
                        else 'lambda-sweep')
    result = lam.measure()
    print('lambda-sweep measure() ->', result)
except Exception as e:
    print(f'(Agent C has not yet written lambda-sweep.py: {e})')
    print('Falling back to inline demo of improvement 103:')
    exec(open(study_dir / 'learning-map/improvements/code/103-lambda-sensitivity-sweep.py').read())


## 5. What to do next

1. **(15 min)** `for f in learning-map/paper/code/*.py; do python "$f"; done`
2. **(2 hr)** Read both proofs in `proofs/`. Goal: write the SDAR loss from memory and derive its gradient unprompted.
3. **(1 day)** Implement improvement 102 in the sandbox and run a short ablation. A negative result is publishable.


---

Generated by Agent D (Stage 7) of the SDAR study run.
